# Unit-diffusion MHM energy estimator

This notebook checks Oswald potential recovery and the complete four-term estimator of [Barrenechea et al.](https://doi.org/10.1137/24M1673073), with identity diffusion and homogeneous Dirichlet data. The reference is the analytical sine problem; the larger figures are archived package calculations, not a published-table equality claim.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import SVG, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "examples"))
from plot_estimator import solve, exact_gradient

estimate = solve(2, 1)
error = estimate.energy_error(exact_gradient, order=12)
assert estimate.total >= error
assert max(estimate.equilibrium_defect) < 1e-10
print({"energy_error": error, "estimator": estimate.total, "effectivity": estimate.total / error})


The recovered potential is a distinct continuous function. Oswald averages count all incident fine triangles and set exterior nodes to zero. The original broken MHM pressure is unchanged. Conservation is against continuous macro-local Pm tests; no fine-cell balance constraint has been added.

In [ ]:
records = json.loads((ROOT / "examples/results/estimator.json").read_text())
for ell in (0, 1):
    rows = [r for r in records["rows"] if r["trace_degree"] == ell]
    assert len(rows) == 5
    assert all(r["effectivity"] >= 1 for r in rows)
    print("trace degree", ell, [(r["macro_resolution"], r["effectivity"]) for r in rows])
for name in ("convergence", "components", "potential", "local-indicators"):
    display(SVG(filename=str(ROOT / "docs/figures/estimator" / f"{name}.svg")))


The estimator includes flux defect, nonconformity, divergence-projection defect and source oscillation. Quadrature-based floating-point evaluation is not an interval-certified bound. Its implementation rejects nonidentity diffusion, nonmatching fine meshes, incompatible degrees and nonzero boundary moments. The energy error uses the actual broken gradient and independent quadrature. An adaptive marking/refinement algorithm and general SPD coefficient bound remain separate extensions.